# Stage 5.2: Pre-test readiness audit

This stage checks whether the Stage 5 development procedure is complete and internally consistent before the held-out evaluation in Stage 6.

The audit uses the training sample, the fixed Stage 4 split and fold specifications, and the saved Stage 5 development outputs. Test-set outcomes and predictors are not used for model evaluation or selection; test identifiers are used only to verify separation from the development sample. Required checks must pass before Stage 6, while review diagnostics are reported for inspection only and do not determine model changes.

## Part 1: Inputs and audit scope

In [1]:
# 1: Import packages and locate the project

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

working_directory = Path.cwd().resolve()

project_root = next(
    (
        directory
        for directory in [working_directory, *working_directory.parents]
        if (
            directory
            / "data_derived"
            / "stage_3_final_modelling_dataset"
            / "final_modelling_dataset.csv"
        ).is_file()
        and (
            directory
            / "data_derived"
            / "stage_5_nested_model_development"
            / "stage_5_outer_selection_register.csv"
        ).is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "The Stage 3 modelling dataset and Stage 5 development files could not be located."
    )

data_derived = project_root / "data_derived"

stage_3_directory = data_derived / "stage_3_final_modelling_dataset"
stage_4_directory = data_derived / "stage_4_modelling_design"
stage_5_directory = data_derived / "stage_5_nested_model_development"
audit_directory = data_derived / "stage_5_pre_test_readiness_audit"
audit_directory.mkdir(parents=True, exist_ok=True)

paths = {
    "modelling data": stage_3_directory / "final_modelling_dataset.csv",
    "split": stage_4_directory / "stage_4_train_test_split.csv",
    "outer folds": stage_4_directory / "stage_4_outer_folds.csv",
    "inner folds": stage_4_directory / "stage_4_inner_folds.csv",
    "predictor roles": stage_4_directory / "stage_4_predictor_preprocessing_roles.csv",
    "model search": stage_4_directory / "stage_4_model_search_specification.csv",
    "candidates": stage_5_directory / "stage_5_hyperparameter_candidates.csv",
    "inner search": stage_5_directory / "stage_5_inner_search_results.csv",
    "selections": stage_5_directory / "stage_5_outer_selection_register.csv",
    "selected predictions": stage_5_directory / "stage_5_selected_outer_predictions.csv",
    "Stage 5 audit": stage_5_directory / "stage_5_final_audit.csv",
}

missing = [name for name, path in paths.items() if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Missing required input file(s): " + ", ".join(missing)
    )

print("Project root located.")
print(
    "Audit output directory: "
    f"{audit_directory.relative_to(project_root).as_posix()}"
)

Project root located.
Audit output directory: data_derived/stage_5_pre_test_readiness_audit


In [2]:
# 2: Load the fixed design and Stage 5 registers

split_assignment = pd.read_csv(
    paths["split"],
    dtype={"NSID": "string"},
)
outer_assignments = pd.read_csv(
    paths["outer folds"],
    dtype={"NSID": "string"},
)
inner_assignments = pd.read_csv(
    paths["inner folds"],
    dtype={"NSID": "string"},
)
predictor_roles = pd.read_csv(paths["predictor roles"])
model_search_specification = pd.read_csv(paths["model search"])
candidate_register = pd.read_csv(paths["candidates"])

# Preserve "None" as the label for unweighted fitting.
inner_search = pd.read_csv(
    paths["inner search"],
    keep_default_na=False,
    na_values=[""],
)
selection_register = pd.read_csv(
    paths["selections"],
    keep_default_na=False,
    na_values=[""],
)
selected_predictions = pd.read_csv(
    paths["selected predictions"],
    dtype={"NSID": "string"},
    keep_default_na=False,
    na_values=[""],
)
stage_5_final_audit = pd.read_csv(paths["Stage 5 audit"])

training_ids = set(
    split_assignment.loc[
        split_assignment["sample"].eq("Training"),
        "NSID",
    ]
)
test_ids = set(
    split_assignment.loc[
        split_assignment["sample"].eq("Test"),
        "NSID",
    ]
)

predictor_columns = predictor_roles["Predictor"].tolist()

modelling_data = pd.read_csv(
    paths["modelling data"],
    dtype={"NSID": "string"},
)
training_data = (
    modelling_data.loc[
        modelling_data["NSID"].isin(training_ids)
    ]
    .copy()
    .reset_index(drop=True)
)
del modelling_data

print(f"Training participants: {len(training_data):,}")
print(f"Predictors: {len(predictor_columns)}")
print(f"Inner-search rows: {len(inner_search):,}")
print(f"Selection rows: {len(selection_register):,}")

Training participants: 7,619
Predictors: 69
Inner-search rows: 1,590
Selection rows: 20


## Part 2: Required integrity and test-isolation checks

These checks verify the development sample, fold assignments, predictor register and Stage 5 outputs before the held-out test sample is opened.

In [3]:
# 3: Run the required integrity checks

expected_models = {"MLR", "RF", "XGBoost", "BRF"}
expected_prediction_models = {
    "Majority-class dummy",
    "MLR",
    "RF",
    "XGBoost",
    "BRF",
}
expected_model_names = {
    "Multinomial logistic regression",
    "Random forest",
    "XGBoost",
    "Balanced random forest",
}
expected_outcome_codes = {1, 4, 5, 6}


def to_boolean_series(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False})
        .fillna(False)
        .astype(bool)
    )


stage_5_audit_passed = (
    "Passed" in stage_5_final_audit.columns
    and to_boolean_series(stage_5_final_audit["Passed"]).all()
)

prediction_counts = selected_predictions.groupby("Model").size()
prediction_unique_ids = (
    selected_predictions.groupby("Model")["NSID"].nunique()
)

inner_fold_structure_ok = True
for outer_fold in range(1, 6):
    outer_validation_ids = set(
        outer_assignments.loc[
            outer_assignments["outer_fold"].eq(outer_fold),
            "NSID",
        ]
    )
    expected_inner_ids = training_ids - outer_validation_ids
    inner_for_outer = inner_assignments.loc[
        inner_assignments["outer_fold"].eq(outer_fold)
    ]

    if (
        set(inner_for_outer["NSID"]) != expected_inner_ids
        or inner_for_outer["NSID"].duplicated().any()
        or set(inner_for_outer["inner_fold"]) != {1, 2, 3, 4}
    ):
        inner_fold_structure_ok = False
        break

hard_checks = {
    "Training sample contains 7,619 participants": len(training_data) == 7_619,
    "Training identifiers are unique": training_data["NSID"].is_unique,
    "Training and test identifiers are disjoint": training_ids.isdisjoint(test_ids),
    "Audit data contain exactly the training identifiers": set(training_data["NSID"]) == training_ids,
    "Audit data contain no test identifiers": set(training_data["NSID"]).isdisjoint(test_ids),
    "All 69 predictors are registered once": (
        len(predictor_roles) == 69
        and predictor_roles["Predictor"].is_unique
    ),
    "All registered predictors are present": set(predictor_columns).issubset(training_data.columns),
    "Outcome contains the fixed four classes": (
        set(training_data["age18_outcome_code"].astype(int))
        == expected_outcome_codes
    ),
    "Outer-fold register contains one assignment per training participant": (
        outer_assignments["NSID"].is_unique
        and set(outer_assignments["NSID"]) == training_ids
        and set(outer_assignments["outer_fold"]) == {1, 2, 3, 4, 5}
    ),
    "Inner-fold registers match each outer-training sample": inner_fold_structure_ok,
    "Stage 4 specifies the four model families": (
        set(model_search_specification["Model"]) == expected_model_names
    ),
    "Macro F1 is the Stage 4 selection metric": (
        model_search_specification["Selection metric"].eq("Macro F1").all()
    ),
    "Stage 5 final audit is fully passed": stage_5_audit_passed,
    "Selected predictions contain the four models and dummy benchmark": (
        set(selected_predictions["Model"]) == expected_prediction_models
    ),
    "Each model has one outer-validation prediction per training participant": all(
        prediction_counts.get(model, 0) == 7_619
        and prediction_unique_ids.get(model, 0) == 7_619
        for model in expected_prediction_models
    ),
    "Selected predictions contain training identifiers only": (
        set(selected_predictions["NSID"]).issubset(training_ids)
    ),
    "Selected predictions contain no test identifiers": (
        set(selected_predictions["NSID"]).isdisjoint(test_ids)
    ),
}

hard_integrity_audit = pd.DataFrame(
    {
        "Check": list(hard_checks.keys()),
        "Passed": list(hard_checks.values()),
    }
)

hard_integrity_path = (
    audit_directory
    / "stage_5_pre_test_hard_integrity_audit.csv"
)
hard_integrity_audit.to_csv(hard_integrity_path, index=False)

display(hard_integrity_audit)

if not hard_integrity_audit["Passed"].all():
    raise AssertionError(
        "A required integrity or test-isolation check failed."
    )

print(f"Saved: {hard_integrity_path.relative_to(project_root)}")

,Check,Passed
0,"Training sample contains 7,619 participants",True
1,Training identifiers are unique,True
2,Training and test identifiers are disjoint,True
3,Audit data contain exactly the training identi...,True
4,Audit data contain no test identifiers,True
5,All 69 predictors are registered once,True
6,All registered predictors are present,True
7,Outcome contains the fixed four classes,True
8,Outer-fold register contains one assignment pe...,True
9,Inner-fold registers match each outer-training...,True


Saved: data_derived\stage_5_pre_test_readiness_audit\stage_5_pre_test_hard_integrity_audit.csv


## Part 3: Search completion and selection consistency

The saved search is checked against the Stage 4 candidate budgets, and each outer-fold selection is reproduced from the saved inner-CV macro-F1 results.

In [4]:
# 4: Check candidate and inner-search completeness

expected_candidate_counts = {
    "MLR": 9,
    "RF": 60,
    "XGBoost": 60,
    "BRF": 60,
}

expected_rows_per_fold = {
    ("MLR", "None"): 9,
    ("MLR", "Balanced"): 9,
    ("RF", "None"): 60,
    ("RF", "Balanced"): 60,
    ("XGBoost", "None"): 60,
    ("XGBoost", "Balanced"): 60,
    ("BRF", "Internal balanced sampling"): 60,
}

expected_search_rows = {
    key: rows_per_fold * 5
    for key, rows_per_fold in expected_rows_per_fold.items()
}

candidate_counts = candidate_register.groupby("Model").size().to_dict()

search_counts = (
    inner_search
    .groupby(["Model", "Weighting mode"])
    .size()
    .to_dict()
)

search_count_detail = (
    inner_search
    .groupby(["Model", "Weighting mode", "Outer fold"])
    .size()
    .to_dict()
)

fold_level_search_complete = all(
    search_count_detail.get((model, mode, fold), 0) == rows_per_fold
    for (model, mode), rows_per_fold in expected_rows_per_fold.items()
    for fold in range(1, 6)
)

inner_fold_score_columns = [
    f"Inner fold {fold} macro F1"
    for fold in range(1, 5)
]

expected_selection_grid = {
    (model, fold)
    for model in expected_models
    for fold in range(1, 6)
}
observed_selection_grid = set(
    zip(
        selection_register["Model"],
        selection_register["Outer fold"],
    )
)

search_completion_checks = {
    "Candidate register contains 189 configurations": len(candidate_register) == 189,
    "Candidate counts match the prespecified budgets": all(
        candidate_counts.get(model, 0) == expected_count
        for model, expected_count in expected_candidate_counts.items()
    ),
    "Inner-search table contains 1,590 candidate summaries": len(inner_search) == 1_590,
    "Inner-search totals match every model and weighting budget": (
        search_counts == expected_search_rows
    ),
    "Each outer fold contains the expected candidates for every fitting mode": (
        fold_level_search_complete
    ),
    "All four inner-fold macro-F1 columns are present": (
        set(inner_fold_score_columns).issubset(inner_search.columns)
    ),
    "No inner-fold macro-F1 score is missing": (
        set(inner_fold_score_columns).issubset(inner_search.columns)
        and inner_search[inner_fold_score_columns].notna().all().all()
    ),
    "Inner macro-F1 means and SDs are complete": (
        inner_search[
            ["Inner macro F1 mean", "Inner macro F1 SD"]
        ].notna().all().all()
    ),
    "Selection register contains the complete model-fold grid": (
        len(selection_register) == 20
        and not selection_register.duplicated(
            ["Model", "Outer fold"]
        ).any()
        and observed_selection_grid == expected_selection_grid
    ),
}

search_completion_audit = pd.DataFrame(
    {
        "Check": list(search_completion_checks.keys()),
        "Passed": list(search_completion_checks.values()),
    }
)

search_completion_path = (
    audit_directory
    / "stage_5_pre_test_search_completion_audit.csv"
)
search_completion_audit.to_csv(
    search_completion_path,
    index=False,
)

display(search_completion_audit)

if not search_completion_audit["Passed"].all():
    raise AssertionError(
        "A required Stage 5 search-completion check failed."
    )

print(f"Saved: {search_completion_path.relative_to(project_root)}")

,Check,Passed
0,Candidate register contains 189 configurations,True
1,Candidate counts match the prespecified budgets,True
2,"Inner-search table contains 1,590 candidate su...",True
3,Inner-search totals match every model and weig...,True
4,Each outer fold contains the expected candidat...,True
5,All four inner-fold macro-F1 columns are present,True
6,No inner-fold macro-F1 score is missing,True
7,Inner macro-F1 means and SDs are complete,True
8,Selection register contains the complete model...,True


Saved: data_derived\stage_5_pre_test_readiness_audit\stage_5_pre_test_search_completion_audit.csv


In [5]:
# 5: Reproduce the saved outer-fold selections from inner-CV macro F1

weighting_priority = {
    "None": 0,
    "Balanced": 1,
    "Internal balanced sampling": 0,
}

selection_reproduction_rows = []

for model_name in ["MLR", "RF", "XGBoost", "BRF"]:
    for outer_fold in range(1, 6):
        fold_rows = inner_search.loc[
            inner_search["Model"].eq(model_name)
            & inner_search["Outer fold"].eq(outer_fold)
        ].copy()

        mode_winners = []

        for weighting_mode, mode_rows in fold_rows.groupby(
            "Weighting mode",
            sort=False,
        ):
            winner = (
                mode_rows
                .sort_values(
                    [
                        "Inner macro F1 mean",
                        "Inner macro F1 SD",
                        "Candidate ID",
                    ],
                    ascending=[False, True, True],
                )
                .iloc[0]
                .copy()
            )
            winner["_weighting_priority"] = (
                weighting_priority[weighting_mode]
            )
            mode_winners.append(winner)

        reproduced = (
            pd.DataFrame(mode_winners)
            .sort_values(
                [
                    "Inner macro F1 mean",
                    "Inner macro F1 SD",
                    "_weighting_priority",
                    "Candidate ID",
                ],
                ascending=[False, True, True, True],
            )
            .iloc[0]
        )

        saved = selection_register.loc[
            selection_register["Model"].eq(model_name)
            & selection_register["Outer fold"].eq(outer_fold)
        ].iloc[0]

        selection_reproduction_rows.append(
            {
                "Model": model_name,
                "Outer fold": outer_fold,
                "Saved weighting mode": saved[
                    "Selected weighting mode"
                ],
                "Reproduced weighting mode": reproduced[
                    "Weighting mode"
                ],
                "Saved candidate ID": int(
                    saved["Selected candidate ID"]
                ),
                "Reproduced candidate ID": int(
                    reproduced["Candidate ID"]
                ),
            }
        )

selection_reproduction = pd.DataFrame(
    selection_reproduction_rows
)

selection_reproduction["Selection matches"] = (
    selection_reproduction["Saved weighting mode"].eq(
        selection_reproduction["Reproduced weighting mode"]
    )
    & selection_reproduction["Saved candidate ID"].eq(
        selection_reproduction["Reproduced candidate ID"]
    )
)

selection_reproduction_path = (
    audit_directory
    / "stage_5_pre_test_selection_reproduction.csv"
)
selection_reproduction.to_csv(
    selection_reproduction_path,
    index=False,
)

display(selection_reproduction)

assert selection_reproduction["Selection matches"].all()

print(
    "All saved selections reproduced:",
    bool(selection_reproduction["Selection matches"].all()),
)
print(
    f"Saved: {selection_reproduction_path.relative_to(project_root)}"
)

,Model,Outer fold,Saved weighting mode,Reproduced weighting mode,Saved candidate ID,Reproduced candidate ID,Selection matches
0,MLR,1,Balanced,Balanced,1,1,True
1,MLR,2,Balanced,Balanced,2,2,True
2,MLR,3,Balanced,Balanced,2,2,True
3,MLR,4,Balanced,Balanced,2,2,True
4,MLR,5,Balanced,Balanced,2,2,True
5,RF,1,Balanced,Balanced,16,16,True
6,RF,2,Balanced,Balanced,5,5,True
7,RF,3,Balanced,Balanced,15,15,True
8,RF,4,Balanced,Balanced,27,27,True
9,RF,5,Balanced,Balanced,5,5,True


All saved selections reproduced: True
Saved: data_derived\stage_5_pre_test_readiness_audit\stage_5_pre_test_selection_reproduction.csv


## Part 4: Search review diagnostics

This diagnostic describes how closely the best inner-CV candidate was separated from nearby candidates. It is a review check rather than a criterion for changing the prespecified search.

In [6]:
# 6: Summarise best-versus-near-best inner-CV performance

search_margin_rows = []

for (model_name, outer_fold, weighting_mode), group in inner_search.groupby(
    ["Model", "Outer fold", "Weighting mode"],
    sort=False,
):
    ranked = (
        group
        .sort_values(
            [
                "Inner macro F1 mean",
                "Inner macro F1 SD",
                "Candidate ID",
            ],
            ascending=[False, True, True],
        )
        .reset_index(drop=True)
    )

    best = ranked.iloc[0]
    second = ranked.iloc[1] if len(ranked) > 1 else None

    best_score = float(best["Inner macro F1 mean"])
    second_score = (
        float(second["Inner macro F1 mean"])
        if second is not None
        else np.nan
    )

    search_margin_rows.append(
        {
            "Model": model_name,
            "Outer fold": int(outer_fold),
            "Weighting mode": weighting_mode,
            "Best candidate ID": int(best["Candidate ID"]),
            "Best inner macro F1": best_score,
            "Second-best inner macro F1": second_score,
            "Best-minus-second margin": (
                best_score - second_score
                if second is not None
                else np.nan
            ),
            "Candidates within 0.005 of best": int(
                (
                    best_score
                    - ranked["Inner macro F1 mean"]
                    <= 0.005 + 1e-12
                ).sum()
            ),
            "Candidates evaluated": len(ranked),
        }
    )

search_margin_summary = pd.DataFrame(search_margin_rows)

assert len(search_margin_summary) == 35

search_margin_path = (
    audit_directory
    / "stage_5_pre_test_search_margin_diagnostics.csv"
)
search_margin_summary.to_csv(
    search_margin_path,
    index=False,
)

close_searches = search_margin_summary.loc[
    search_margin_summary[
        "Candidates within 0.005 of best"
    ].gt(1)
].copy()

display(close_searches.round(5))
print(
    "Searches with more than one candidate within 0.005 of best: "
    f"{len(close_searches)} of {len(search_margin_summary)}"
)
print(f"Saved: {search_margin_path.relative_to(project_root)}")

,Model,Outer fold,Weighting mode,Best candidate ID,Best inner macro F1,Second-best inner macro F1,Best-minus-second margin,Candidates within 0.005 of best,Candidates evaluated
0,MLR,1,None,5,0.38607,0.38598,0.00009,7,9
1,MLR,1,Balanced,1,0.39388,0.39283,0.00105,9,9
2,MLR,2,None,6,0.39717,0.39639,0.00077,5,9
3,MLR,2,Balanced,2,0.40444,0.39981,0.00464,2,9
4,MLR,3,None,9,0.38851,0.38822,0.00029,7,9
5,MLR,3,Balanced,2,0.39238,0.39001,0.00237,4,9
6,MLR,4,None,8,0.39866,0.39828,0.00038,6,9
7,MLR,4,Balanced,2,0.40474,0.40418,0.00055,3,9
8,MLR,5,None,6,0.38378,0.38374,0.00003,6,9
9,MLR,5,Balanced,2,0.40026,0.39969,0.00057,5,9


Searches with more than one candidate within 0.005 of best: 32 of 35
Saved: data_derived\stage_5_pre_test_readiness_audit\stage_5_pre_test_search_margin_diagnostics.csv


## Part 5: MLR fitting diagnostics

The five fold-specific MLR specifications selected in Stage 5 are fitted again on their corresponding outer-training samples. This diagnostic refit does not change the selected configuration and does not use Stage 6 data.

In [7]:
# 7: Define the Stage 5 MLR pipeline

numeric_predictors = predictor_roles.loc[
    predictor_roles["Preprocessing role"].eq("Numeric"),
    "Predictor",
].tolist()

categorical_predictors = predictor_roles.loc[
    predictor_roles["Preprocessing role"].eq("Categorical"),
    "Predictor",
].tolist()

assert len(numeric_predictors) + len(categorical_predictors) == 69
assert set(numeric_predictors).isdisjoint(categorical_predictors)

for predictor in numeric_predictors:
    training_data[predictor] = pd.to_numeric(
        training_data[predictor],
        errors="raise",
    )

outcome_code_to_internal = {1: 0, 4: 1, 5: 2, 6: 3}
training_data["model_outcome"] = (
    training_data["age18_outcome_code"]
    .map(outcome_code_to_internal)
    .astype(int)
)


def make_mlr_pipeline(C, weighting_mode):
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    drop="if_binary",
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_predictors),
            ("categorical", categorical_pipeline, categorical_predictors),
        ],
        remainder="drop",
    )

    estimator = LogisticRegression(
        C=float(C),
        penalty="l2",
        solver="lbfgs",
        max_iter=3000,
        class_weight="balanced" if weighting_mode == "Balanced" else None,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ]
    )


print(f"Numeric/ordinal predictors: {len(numeric_predictors)}")
print(f"Binary/nominal predictors: {len(categorical_predictors)}")

Numeric/ordinal predictors: 38
Binary/nominal predictors: 31


In [8]:
# 8: Refit the selected MLR specifications and capture fitting diagnostics

mlr_diagnostic_rows = []

for outer_fold in range(1, 6):
    selected = selection_register.loc[
        selection_register["Model"].eq("MLR")
        & selection_register["Outer fold"].eq(outer_fold)
    ].iloc[0]

    parameters = json.loads(selected["Selected parameters"])
    weighting_mode = selected["Selected weighting mode"]

    outer_validation_ids = set(
        outer_assignments.loc[
            outer_assignments["outer_fold"].eq(outer_fold),
            "NSID",
        ]
    )

    outer_training = training_data.loc[
        ~training_data["NSID"].isin(outer_validation_ids)
    ].copy()

    pipeline = make_mlr_pipeline(
        C=parameters["C"],
        weighting_mode=weighting_mode,
    )

    with warnings.catch_warnings(record=True) as captured_warnings:
        warnings.simplefilter("always", ConvergenceWarning)
        pipeline.fit(
            outer_training[predictor_columns],
            outer_training["model_outcome"],
        )

    convergence_messages = [
        str(item.message)
        for item in captured_warnings
        if issubclass(item.category, ConvergenceWarning)
    ]

    model = pipeline.named_steps["model"]
    max_iteration_used = int(np.max(model.n_iter_))
    coefficients = np.asarray(model.coef_)

    mlr_diagnostic_rows.append(
        {
            "Outer fold": outer_fold,
            "Selected C": float(parameters["C"]),
            "Selected weighting mode": weighting_mode,
            "Outer-training participants": len(outer_training),
            "Transformed features": int(coefficients.shape[1]),
            "Maximum iterations used": max_iteration_used,
            "Maximum iterations allowed": int(model.max_iter),
            "Convergence warning": bool(convergence_messages),
            "Warning message": " | ".join(convergence_messages),
            "Maximum absolute coefficient": float(np.max(np.abs(coefficients))),
            "All coefficients finite": bool(np.isfinite(coefficients).all()),
        }
    )

mlr_convergence_diagnostics = pd.DataFrame(mlr_diagnostic_rows)
mlr_convergence_diagnostics["Iteration limit reached"] = (
    mlr_convergence_diagnostics["Maximum iterations used"]
    >= mlr_convergence_diagnostics["Maximum iterations allowed"]
)
mlr_convergence_diagnostics["Hard failure"] = (
    mlr_convergence_diagnostics["Convergence warning"]
    | mlr_convergence_diagnostics["Iteration limit reached"]
    | ~mlr_convergence_diagnostics["All coefficients finite"]
)

mlr_convergence_path = audit_directory / "stage_5_pre_test_mlr_convergence_diagnostics.csv"
mlr_convergence_diagnostics.to_csv(mlr_convergence_path, index=False)

display(mlr_convergence_diagnostics)
print(f"Saved: {mlr_convergence_path.relative_to(project_root)}")

,Outer fold,Selected C,Selected weighting mode,Outer-training participants,Transformed features,Maximum iterations used,Maximum iterations allowed,Convergence warning,Warning message,Maximum absolute coefficient,All coefficients finite,Iteration limit reached,Hard failure
0,1,0.01,Balanced,6095,118,52,3000,False,,0.276530,True,False,False
1,2,0.03,Balanced,6095,118,77,3000,False,,0.401650,True,False,False
2,3,0.03,Balanced,6095,118,72,3000,False,,0.403878,True,False,False
3,4,0.03,Balanced,6095,118,75,3000,False,,0.501731,True,False,False
4,5,0.03,Balanced,6096,118,75,3000,False,,0.478140,True,False,False


Saved: data_derived\stage_5_pre_test_readiness_audit\stage_5_pre_test_mlr_convergence_diagnostics.csv


## Part 6: Categorical-level sparsity diagnostics

Observed categorical levels are checked within each outer-training sample. Sparse or zero destination cells can make individual encoded effects less stable, but they do not by themselves justify recoding a source category.

In [9]:
# 9: Count categorical levels within each outer-training sample

class_codes = [1, 4, 5, 6]
class_names = {
    1: "Education",
    4: "Employment",
    5: "Apprenticeship or training",
    6: "Unemployment or inactivity (NEET)",
}

categorical_level_rows = []

for outer_fold in range(1, 6):
    outer_validation_ids = set(
        outer_assignments.loc[
            outer_assignments["outer_fold"].eq(outer_fold),
            "NSID",
        ]
    )
    outer_training = training_data.loc[
        ~training_data["NSID"].isin(outer_validation_ids)
    ].copy()

    for predictor in categorical_predictors:
        local = outer_training[[predictor, "age18_outcome_code"]].copy()
        local = local.loc[local[predictor].notna()].copy()
        local["Level"] = local[predictor].astype("string")

        if local.empty:
            continue

        total_counts = local["Level"].value_counts(dropna=False)
        class_count_table = (
            local.groupby(["Level", "age18_outcome_code"])
            .size()
            .unstack(fill_value=0)
            .reindex(columns=class_codes, fill_value=0)
        )

        for level, total_count in total_counts.items():
            class_counts = class_count_table.loc[level]
            row = {
                "Outer fold": outer_fold,
                "Predictor": predictor,
                "Level": str(level),
                "Observed count": int(total_count),
                "Observed proportion": float(total_count / len(outer_training)),
                "Sparse total (<20)": int(total_count) < 20,
                "Zero in any destination": bool((class_counts == 0).any()),
            }
            for code_value in class_codes:
                row[f"{class_names[code_value]} count"] = int(class_counts[code_value])
            categorical_level_rows.append(row)

categorical_level_diagnostics = pd.DataFrame(categorical_level_rows)

categorical_predictor_summary = (
    categorical_level_diagnostics.groupby(["Outer fold", "Predictor"], as_index=False)
    .agg(
        observed_levels=("Level", "nunique"),
        minimum_level_count=("Observed count", "min"),
        sparse_levels=("Sparse total (<20)", "sum"),
        levels_with_zero_destination=("Zero in any destination", "sum"),
    )
)

categorical_level_path = audit_directory / "stage_5_pre_test_categorical_level_diagnostics.csv"
categorical_summary_path = audit_directory / "stage_5_pre_test_categorical_predictor_summary.csv"
categorical_level_diagnostics.to_csv(categorical_level_path, index=False)
categorical_predictor_summary.to_csv(categorical_summary_path, index=False)

flagged_categorical_predictors = categorical_predictor_summary.loc[
    categorical_predictor_summary["sparse_levels"].gt(0)
    | categorical_predictor_summary["levels_with_zero_destination"].gt(0)
].copy()

print("Categorical predictors with at least one review flag:")
display(flagged_categorical_predictors)
print(
    "Sparse level-fold rows:",
    int(categorical_level_diagnostics["Sparse total (<20)"].sum()),
)
print(
    "Level-fold rows with a zero destination cell:",
    int(categorical_level_diagnostics["Zero in any destination"].sum()),
)
print(f"Saved: {categorical_level_path.relative_to(project_root)}")
print(f"Saved: {categorical_summary_path.relative_to(project_root)}")

Categorical predictors with at least one review flag:


,Outer fold,Predictor,observed_levels,minimum_level_count,sparse_levels,levels_with_zero_destination
12,1,family_composition,5,55,0,1
17,1,housing_tenure,8,20,0,1
48,2,housing_tenure,8,23,0,1
79,3,housing_tenure,8,19,1,2
110,4,housing_tenure,8,17,1,1
141,5,housing_tenure,8,21,0,1


Sparse level-fold rows: 2
Level-fold rows with a zero destination cell: 7
Saved: data_derived\stage_5_pre_test_readiness_audit\stage_5_pre_test_categorical_level_diagnostics.csv
Saved: data_derived\stage_5_pre_test_readiness_audit\stage_5_pre_test_categorical_predictor_summary.csv


## Part 7: Readiness decision

The readiness decision is based on the required checks. Narrow score margins and sparse categorical levels remain review diagnostics. 

In [10]:
# 10: Combine required checks and review diagnostics

selection_check = bool(
    selection_reproduction["Selection matches"].all()
)
mlr_check = bool(
    ~mlr_convergence_diagnostics["Hard failure"].any()
)

hard_checks_combined = pd.concat(
    [
        hard_integrity_audit.assign(
            Area="Integrity and isolation"
        ),
        search_completion_audit.assign(
            Area="Search completion"
        ),
        pd.DataFrame(
            [
                {
                    "Check": (
                        "Saved outer-fold selections reproduce "
                        "from inner-CV macro F1"
                    ),
                    "Passed": selection_check,
                    "Area": "Selection consistency",
                },
                {
                    "Check": (
                        "Selected MLR fits converge without "
                        "hard diagnostics"
                    ),
                    "Passed": mlr_check,
                    "Area": "MLR fitting",
                },
            ]
        ),
    ],
    ignore_index=True,
)

hard_checks_combined = hard_checks_combined[
    ["Area", "Check", "Passed"]
]

review_summary = pd.DataFrame(
    [
        {
            "Diagnostic": (
                "Searches with more than one candidate "
                "within 0.005 of best"
            ),
            "Flagged rows": int(len(close_searches)),
            "Interpretation": (
                "Review only; close candidate scores indicate "
                "limited separation within the searched set."
            ),
        },
        {
            "Diagnostic": (
                "Categorical level-fold rows with "
                "observed count below 20"
            ),
            "Flagged rows": int(
                categorical_level_diagnostics[
                    "Sparse total (<20)"
                ].sum()
            ),
            "Interpretation": (
                "Review only; sparse observed levels do not "
                "automatically justify recoding."
            ),
        },
        {
            "Diagnostic": (
                "Categorical level-fold rows with "
                "a zero destination cell"
            ),
            "Flagged rows": int(
                categorical_level_diagnostics[
                    "Zero in any destination"
                ].sum()
            ),
            "Interpretation": (
                "Review only; zero cells are descriptive and "
                "do not automatically require a model change."
            ),
        },
    ]
)

hard_checks_path = (
    audit_directory
    / "stage_5_pre_test_final_hard_checks.csv"
)
review_summary_path = (
    audit_directory
    / "stage_5_pre_test_review_summary.csv"
)

hard_checks_combined.to_csv(
    hard_checks_path,
    index=False,
)
review_summary.to_csv(
    review_summary_path,
    index=False,
)

print("Required checks:")
display(hard_checks_combined)

print("\nReview diagnostics:")
display(review_summary)

all_hard_checks_pass = bool(
    hard_checks_combined["Passed"].all()
)

print("\nReadiness status:")
if all_hard_checks_pass:
    print(
        "PASS: No required readiness check failed. "
        "Review the diagnostic results before Stage 6."
    )
else:
    print(
        "STOP: Resolve the failed required check before Stage 6."
    )

Required checks:


,Area,Check,Passed
0,Integrity and isolation,"Training sample contains 7,619 participants",True
1,Integrity and isolation,Training identifiers are unique,True
2,Integrity and isolation,Training and test identifiers are disjoint,True
3,Integrity and isolation,Audit data contain exactly the training identi...,True
4,Integrity and isolation,Audit data contain no test identifiers,True
5,Integrity and isolation,All 69 predictors are registered once,True
6,Integrity and isolation,All registered predictors are present,True
7,Integrity and isolation,Outcome contains the fixed four classes,True
8,Integrity and isolation,Outer-fold register contains one assignment pe...,True
9,Integrity and isolation,Inner-fold registers match each outer-training...,True



Review diagnostics:


,Diagnostic,Flagged rows,Interpretation
0,Searches with more than one candidate within 0...,32,Review only; close candidate scores indicate l...
1,Categorical level-fold rows with observed coun...,2,Review only; sparse observed levels do not aut...
2,Categorical level-fold rows with a zero destin...,7,Review only; zero cells are descriptive and do...



Readiness status:
PASS: No required readiness check failed. Review the diagnostic results before Stage 6.


## Part 8: Output manifest and completion

In [11]:
# 11: Save the output manifest and completion record

output_files = [
    "stage_5_pre_test_hard_integrity_audit.csv",
    "stage_5_pre_test_search_completion_audit.csv",
    "stage_5_pre_test_selection_reproduction.csv",
    "stage_5_pre_test_search_margin_diagnostics.csv",
    "stage_5_pre_test_mlr_convergence_diagnostics.csv",
    "stage_5_pre_test_categorical_level_diagnostics.csv",
    "stage_5_pre_test_categorical_predictor_summary.csv",
    "stage_5_pre_test_final_hard_checks.csv",
    "stage_5_pre_test_review_summary.csv",
]

output_manifest = pd.DataFrame(
    {
        "File": output_files,
        "Path": [
            str(
                (audit_directory / filename)
                .relative_to(project_root)
            )
            for filename in output_files
        ],
        "Exists": [
            (audit_directory / filename).is_file()
            for filename in output_files
        ],
    }
)

output_manifest_path = (
    audit_directory
    / "stage_5_pre_test_output_manifest.csv"
)
output_manifest.to_csv(
    output_manifest_path,
    index=False,
)

completion_record = pd.DataFrame(
    [
        {
            "Required checks passed": bool(
                hard_checks_combined["Passed"].all()
            ),
            "Required checks": len(hard_checks_combined),
            "Failed required checks": int(
                (~hard_checks_combined["Passed"]).sum()
            ),
            "Review diagnostics": len(review_summary),
            "Output files present": bool(
                output_manifest["Exists"].all()
            ),
        }
    ]
)

completion_path = (
    audit_directory
    / "stage_5_pre_test_completion.csv"
)
completion_record.to_csv(
    completion_path,
    index=False,
)

print("Output manifest:")
display(output_manifest)

print("\nCompletion record:")
display(completion_record)

print(
    f"\nSaved: {output_manifest_path.relative_to(project_root)}"
)
print(
    f"Saved: {completion_path.relative_to(project_root)}"
)

Output manifest:


,File,Path,Exists
0,stage_5_pre_test_hard_integrity_audit.csv,data_derived\stage_5_pre_test_readiness_audit\...,True
1,stage_5_pre_test_search_completion_audit.csv,data_derived\stage_5_pre_test_readiness_audit\...,True
2,stage_5_pre_test_selection_reproduction.csv,data_derived\stage_5_pre_test_readiness_audit\...,True
3,stage_5_pre_test_search_margin_diagnostics.csv,data_derived\stage_5_pre_test_readiness_audit\...,True
4,stage_5_pre_test_mlr_convergence_diagnostics.csv,data_derived\stage_5_pre_test_readiness_audit\...,True
5,stage_5_pre_test_categorical_level_diagnostics...,data_derived\stage_5_pre_test_readiness_audit\...,True
6,stage_5_pre_test_categorical_predictor_summary...,data_derived\stage_5_pre_test_readiness_audit\...,True
7,stage_5_pre_test_final_hard_checks.csv,data_derived\stage_5_pre_test_readiness_audit\...,True
8,stage_5_pre_test_review_summary.csv,data_derived\stage_5_pre_test_readiness_audit\...,True



Completion record:


,Required checks passed,Required checks,Failed required checks,Review diagnostics,Output files present
0,True,28,0,3,True



Saved: data_derived\stage_5_pre_test_readiness_audit\stage_5_pre_test_output_manifest.csv
Saved: data_derived\stage_5_pre_test_readiness_audit\stage_5_pre_test_completion.csv


## Stage 5.2 summary

The pre-test audit verifies the fixed training sample, fold structure, search completion, saved outer-fold selections and MLR fitting before the held-out test evaluation. Required checks must pass before Stage 6, while review diagnostics are inspected separately and do not determine model changes.